# Semaine 5 — Jour 4 : PostgreSQL pour systèmes IA en production

Ce notebook étudiant accompagne les fichiers Markdown du jour.

Objectif : comprendre et pratiquer la conception d’une couche PostgreSQL durable pour un assistant IA de production.


## 1. Redis vs PostgreSQL

Redis accélère les accès volatils : cache, sessions courtes, rate limit, verrous.

PostgreSQL conserve les preuves durables : conversations, messages, runs, tool calls, évaluations, audit.

Règle de conception :

> Redis accélère. PostgreSQL prouve.


## 2. Architecture cible

```mermaid
flowchart LR
    API[FastAPI] --> AG[Agent Service]
    AG --> REDIS[(Redis)]
    AG --> PG[(PostgreSQL)]
    AG --> LLM[LLM Provider]
    AG --> TOOLS[Tools]
    PG --> OBS[Monitoring]
    PG --> EVAL[Evaluation]
```


## 3. Entités à modéliser

- `Conversation` : fil durable d’interaction.
- `Message` : contenu utilisateur, assistant, système ou outil.
- `AgentRun` : exécution interne de l’agent.
- `ToolCall` : action externe appelée par l’agent.
- `Evaluation` : mesure de qualité associée à un run.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import StrEnum
from uuid import uuid4


class MessageRole(StrEnum):
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"


class RunStatus(StrEnum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


def new_id() -> str:
    return str(uuid4())


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


@dataclass(frozen=True)
class Conversation:
    tenant_id: str
    user_id: str
    title: str
    id: str = field(default_factory=new_id)
    created_at: datetime = field(default_factory=utc_now)


@dataclass(frozen=True)
class Message:
    conversation_id: str
    role: MessageRole
    content: str
    id: str = field(default_factory=new_id)
    created_at: datetime = field(default_factory=utc_now)


## 4. Mini-exercice — créer une conversation

Complétez la cellule suivante pour créer une conversation, ajouter deux messages et les afficher dans l’ordre.


In [ ]:
conversation = Conversation(
    tenant_id="tenant-acme",
    user_id="user-001",
    title="Diagnostic incident"
)

messages: list[Message] = []

# TODO: ajouter un message utilisateur
# TODO: ajouter une réponse assistant
# TODO: afficher les rôles et contenus


## 5. Schéma SQL minimal

Analysez le schéma suivant. Identifiez les contraintes d’intégrité et expliquez leur rôle.


In [ ]:
schema_sql = '''
create table conversations (
    id uuid primary key,
    tenant_id text not null,
    user_id text not null,
    title text not null,
    status text not null check (status in ('active', 'archived')),
    metadata jsonb not null default '{}'::jsonb,
    created_at timestamptz not null default now(),
    updated_at timestamptz not null default now()
);

create table messages (
    id uuid primary key,
    conversation_id uuid not null references conversations(id) on delete cascade,
    role text not null check (role in ('system', 'user', 'assistant', 'tool')),
    content text not null,
    metadata jsonb not null default '{}'::jsonb,
    created_at timestamptz not null default now()
);
'''
print(schema_sql)


## 6. Exercice — choisir les index

Pour chaque requête, proposez un index :

```sql
select *
from messages
where conversation_id = $1
order by created_at asc;
```

```sql
select *
from agent_runs
where status = 'failed'
order by started_at desc
limit 50;
```


In [ ]:
# TODO: écrire les index SQL sous forme de chaînes Python.
index_messages = """"""
index_failed_runs = """"""

print(index_messages)
print(index_failed_runs)


## 7. Repository pattern

Le repository masque les détails de PostgreSQL au code métier.


In [ ]:
from typing import Protocol


class ConversationStore(Protocol):
    def create_conversation(self, tenant_id: str, user_id: str, title: str) -> Conversation:
        ...

    def append_message(self, conversation_id: str, role: MessageRole, content: str) -> Message:
        ...

    def list_messages(self, conversation_id: str) -> list[Message]:
        ...


## 8. Travail final

Réalisez le challenge du jour :

- proposer un schéma complet ;
- ajouter les index ;
- définir une stratégie transactionnelle ;
- expliquer la séparation Redis / PostgreSQL ;
- écrire un repository testable.
